<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/Projects/05-PersonalChefAgent/Chef_Agent(test_build).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You're gonna create a personal chef assistant who, takes the leftover ingredients you have in your kitchen, searches the web for recipes that use those ingredients, returns them to you. And can answer any follow up questions you have. .i.e "has memory" of the previous runs.

Now you should be able to do this project using just the techniques we've already covered so far. Bonus for those who watch the multimodal messages video. See if you can upload an image of your fridge or pantry and get it to pick out a recipe based off that.


In [1]:
# Clone the repo
! git clone --depth 1 https://github.com/langchain-ai/lca-lc-foundations.git

Cloning into 'lca-lc-foundations'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 140 (delta 27), reused 85 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.09 MiB | 2.67 MiB/s, done.
Resolving deltas: 100% (27/27), done.


## Setup and imports

In [2]:
!pip install -q langchain_groq tavily

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00


In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_groq import ChatGroq
from google.colab import userdata
from tavily import TavilyClient
from ipywidgets import FileUpload
from IPython.display import display
import base64
from langgraph.checkpoint.memory import InMemorySaver

In [5]:
TAVILY_API_KEY = userdata.get('tavily_key')
tavily_client = TavilyClient(TAVILY_API_KEY)

In [6]:
@tool
def web_search(query: str) -> str:
    """Searches the web for the given input query."""
    response = tavily_client.search(query)
    if response.get('answer'):
        return response['answer']
    elif 'results' in response:
        # Concatenate content of top results as a fallback
        return " ".join([r['content'] for r in response['results'] if 'content' in r][:3])
    else:
        return "No relevant information found."

In [7]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get('groq_key'),
    temperature=0.1,
)

In [8]:
llm_image = ChatGroq(
    model="qwen/qwen3.6-27b",
    api_key=userdata.get('groq_key'),
    temperature=0.1,
)

In [9]:
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)


FileUpload(value={}, accept='.png', description='Upload')

In [12]:
if not uploader.value:
    print("Please upload an image using the widget above before running this cell.")
else:
    # Get the filename (first key) from the uploader.value dictionary
    filename = next(iter(uploader.value))

    # Get the uploaded file dict using the filename
    uploaded_file = uploader.value[filename]

    # This is a memoryview
    content_mv = uploaded_file["content"]

    # Convert memoryview -> bytes
    img_bytes = bytes(content_mv)  # or content_mv.tobytes()

    # Now base64 encode
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    print("Image processed successfully!")

Image processed successfully!


In [13]:
image_agent= create_agent(model=llm_image,
                          system_prompt="Identify all food items and ingredients visible in the image. Your response MUST be a comma-separated list of these items, and ONLY these items. Do NOT include any conversational text, introductory phrases, explanations, or any form of thought process. Strictly adhere to outputting just the comma-separated list.")

In [15]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Identify all food items and ingredients visible in the image. Your response MUST be a comma-separated list of these items, and ONLY these items. Do NOT include any conversational text, introductory phrases, explanations, or any form of thought process. Strictly adhere to outputting just the comma-separated list. For example: apple, banana, orange."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])
response = image_agent.invoke({"messages":[multimodal_question]})


In [25]:
from pprint import pprint


In [18]:
ingredients_agent= create_agent(model=llm,
                                system_prompt="extract and output only a comma separated list of ingredients from the provided text.")


In [21]:
ingredients= ingredients_agent.invoke({"messages":[HumanMessage(content=response["messages"][-1].content)]})
pprint(ingredients)

{'messages': [HumanMessage(content='\n<think>\nThe user wants me to identify food items in the provided image.\nI need to scan the image, which shows the inside of a refrigerator, and list all visible food items.\nThe output must be a simple comma-separated list.\n\n1.  **Top Shelf (Left to Right):**\n    *   Glass bottles with liquid (water/juice).\n    *   Cucumber slices in a bottle.\n    *   Milk bottle.\n    *   Strawberries in a green basket.\n    *   Containers on the right side:\n        *   Yellow peppers/chopped peppers.\n        *   Corn kernels.\n        *   Red onions (sliced).\n        *   Fish fillets (white fish).\n        *   Green vegetables (looks like chopped broccoli or green beans).\n        *   Pasta or noodles.\n\n2.  **Middle Shelf (Left to Right):**\n    *   Bottle of orange juice.\n    *   Bottle of dark liquid (vinegar/oil/wine).\n    *   Green leafy herbs (parsley/cilantro).\n    *   Milk jug.\n    *   Jar of pickles.\n    *   Green beans.\n    *   Radishes

In [23]:
chef_system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""
chef_agent= create_agent(model=llm,
                         tools=[web_search],
                         system_prompt=chef_system_prompt,
                         checkpointer=InMemorySaver())

In [24]:
config = {"configurable": {"thread_id": "1"}}

response = chef_agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Here are some recipe suggestions using leftover chicken and rice:

1. Chicken Fried Rice: A simple and delicious one-pan dinner made with leftover chicken, rice, and frozen peas and carrots.
2. Jambalaya: A spicy dish made with leftover chicken, rice, tomato paste, Rotel, Cajun seasoning, and shrimp (optional).
3. Leftover Chicken and Egg Fried Rice: A family-friendly weeknight dinner made with leftover chicken, rice, eggs, and soy sauce.

Would you like the recipe instructions for any of these options?
